# Fluxo Clínico com LangGraph, Logging, Explainability e Validação Humana

Este notebook implementa a etapa final do assistente médico da **Fase 3**.  
O foco aqui é demonstrar que o sistema não apenas responde perguntas clínicas, mas também opera com:

- **fluxo estruturado com LangGraph**
- **contextualização com dados do paciente**
- **logging e auditoria**
- **explicabilidade da resposta**
- **validação humana antes da recomendação final**

> **Observação:** este notebook foi construído para ser didático e avaliável.  
> Quando as credenciais do Azure OpenAI estiverem disponíveis, ele utiliza o modelo configurado.  
> Caso contrário, ele executa um **modo de demonstração** com resposta simulada, preservando o fluxo completo para avaliação.

## Objetivo do notebook

Demonstrar um fluxo clínico automatizado e seguro, onde o sistema:

1. recebe os dados do paciente
2. realiza uma triagem inicial
3. verifica exames pendentes
4. consulta o assistente médico
5. gera uma resposta com **justificativa**
6. passa por uma etapa de **validação humana**
7. registra a execução em **log estruturado**

## Bibliotecas utilizadas

Neste notebook utilizamos:

- `LangChain` para construção da chain de interação com a LLM
- `LangGraph` para orquestrar o fluxo
- `dotenv` para leitura de variáveis de ambiente
- `json` e `pathlib` para persistência dos logs
- `typing` para definição do estado clínico

> Caso alguma biblioteca não esteja instalada no ambiente local, ajuste o `requirements.txt` do projeto antes da execução.

In [1]:
import os
import json
from pathlib import Path
from datetime import datetime
from typing import TypedDict, List, Dict, Any, Optional

from dotenv import load_dotenv

from langchain_core.prompts import ChatPromptTemplate
from langgraph.graph import StateGraph, END

try:
    from langchain_openai import AzureChatOpenAI
    AZURE_OPENAI_AVAILABLE = True
except Exception:
    AZURE_OPENAI_AVAILABLE = False

load_dotenv()

True

## Inicialização do modelo

A célula abaixo tenta conectar ao modelo fine-tuned configurado no Azure OpenAI.

- Se as variáveis de ambiente estiverem corretas, o notebook usa o modelo real.
- Se não estiverem, ele usa um **modelo simulado**, permitindo que o professor avalie toda a lógica do fluxo mesmo sem acesso à infraestrutura.

In [ ]:
class MockResponse:
    def __init__(self, content: str):
        self.content = content


class MockLLM:
    """Modelo simulado para demonstração quando o Azure OpenAI não estiver disponível."""

    def invoke(self, payload: Dict[str, Any]) -> MockResponse:
        patient_data = payload.get("patient_data", {})
        question = payload.get("question", "")

        sintomas = patient_data.get("sintomas", [])
        exames = patient_data.get("exames_pendentes", [])
        idade = patient_data.get("idade", "não informado")

        resposta = f"""
Análise clínica inicial:
Paciente com {idade} anos apresentando os sintomas {sintomas}.

Sugestão clínica:
- revisar sinais vitais e histórico recente
- considerar investigação complementar de causas infecciosas ou cardiopulmonares conforme o quadro
- priorizar a avaliação médica presencial em caso de piora clínica

Exames pendentes considerados:
{exames if exames else 'nenhum exame pendente informado'}

Justificativa:
A recomendação foi construída a partir dos sintomas informados, dos exames pendentes e das regras clínicas definidas no fluxo.

Observação de segurança:
Esta resposta não substitui avaliação médica e deve ser validada por profissional responsável.

Pergunta original:
{question}
""".strip()

        return MockResponse(resposta)


def build_llm():
    deployment = os.getenv("AZURE_OPENAI_DEPLOYMENT_NAME")
    api_version = os.getenv("AZURE_API_VERSION")

    if AZURE_OPENAI_AVAILABLE and deployment and api_version:
        print("Azure OpenAI configurado. Utilizando modelo real.")
        return AzureChatOpenAI(
            azure_deployment=deployment,
            api_version=api_version,
            temperature=0
        )

    print("Azure OpenAI não configurado. Utilizando modo de demonstração com MockLLM.")
    return MockLLM()


llm = build_llm()

✅ Azure OpenAI configurado. Utilizando modelo real.


## Prompt clínico do assistente

O prompt abaixo define o comportamento esperado da LLM dentro do fluxo.  
Algumas regras foram mantidas de forma explícita para reforçar a segurança do sistema:

- não prescrever medicamentos diretamente
- sugerir apenas próximos passos, exames ou hipóteses iniciais
- indicar que a decisão final depende de validação humana
- justificar a recomendação com base no contexto recebido

In [3]:
prompt = ChatPromptTemplate.from_template("""
Você é um assistente médico treinado com dados clínicos e literatura científica.

Contexto do paciente:
{patient_data}

Pergunta do médico:
{question}

Regras obrigatórias:
- Não prescrever medicamentos diretamente.
- Não substituir decisão médica humana.
- Sugerir apenas próximos passos, exames, hipóteses iniciais ou condutas gerais.
- Explicar brevemente a lógica utilizada para chegar à recomendação.
- Informar de maneira explícita que a decisão final deve ser validada por um profissional de saúde.

Estruture sua resposta com os blocos:
1. Análise clínica inicial
2. Sugestão clínica
3. Justificativa
4. Observação de segurança
""")

chain = prompt | llm

## Definição do estado clínico

O estado abaixo representa as informações que trafegam entre os nós do LangGraph.

Ele foi modelado para deixar evidente:
- os dados clínicos de entrada
- o nível de risco
- a resposta gerada pela LLM
- a explicação da resposta
- a validação humana
- o caminho do arquivo de log gerado

In [4]:
class MedicalState(TypedDict, total=False):
    patient_id: str
    patient_data: Dict[str, Any]
    question: str
    risk_level: str
    pending_exams: List[str]
    llm_response: str
    structured_answer: Dict[str, Any]
    human_validation: Dict[str, Any]
    final_output: Dict[str, Any]
    log_path: str

## Logging estruturado e auditoria

Cada execução gera um registro estruturado em JSONL, contendo:

- identificador do paciente
- timestamp
- classificação de risco
- exames pendentes
- resposta da LLM
- justificativa
- resultado da validação humana

In [6]:
LOG_DIR = Path("04_logs")
LOG_DIR.mkdir(exist_ok=True)
LOG_FILE = LOG_DIR / "logs_clinicos.jsonl"


def append_log(event: Dict[str, Any], file_path: Path = LOG_FILE) -> str:
    event["timestamp"] = datetime.utcnow().isoformat() + "Z"

    with open(file_path, "a", encoding="utf-8") as f:
        f.write(json.dumps(event, ensure_ascii=False) + "\n")

    return str(file_path)

## Implementação dos nós do fluxo

A seguir implementamos cada etapa do processo clínico.

### Etapas:
1. **Triagem clínica**: classifica o risco inicial
2. **Verificação de exames**: recupera exames pendentes
3. **Consulta à LLM**: obtém a recomendação inicial
4. **Explainability**: estrutura a resposta e explicita a justificativa
5. **Validação humana**: simula a aprovação por um profissional
6. **Registro em log**: persiste a execução para auditoria

In [7]:
def clinical_triage(state: MedicalState) -> Dict[str, Any]:
    symptoms = [s.lower() for s in state["patient_data"].get("sintomas", [])]

    high_risk_symptoms = {
        "dor no peito",
        "falta de ar",
        "perda de consciência",
        "confusão mental",
        "saturação baixa",
    }

    moderate_risk_symptoms = {
        "febre persistente",
        "fadiga intensa",
        "taquicardia",
        "tontura",
    }

    if any(symptom in high_risk_symptoms for symptom in symptoms):
        risk = "alto"
    elif any(symptom in moderate_risk_symptoms for symptom in symptoms):
        risk = "moderado"
    else:
        risk = "baixo"

    return {"risk_level": risk}


def check_pending_exams(state: MedicalState) -> Dict[str, Any]:
    pending = state["patient_data"].get("exames_pendentes", [])
    return {"pending_exams": pending}


def ask_medical_assistant(state: MedicalState) -> Dict[str, Any]:
    response = chain.invoke({
        "question": state["question"],
        "patient_data": state["patient_data"]
    })

    content = response.content if hasattr(response, "content") else str(response)
    return {"llm_response": content}


def build_explainability(state: MedicalState) -> Dict[str, Any]:
    patient_data = state["patient_data"]
    sintomas = patient_data.get("sintomas", [])
    sinais_vitais = patient_data.get("sinais_vitais", {})
    pending_exams = state.get("pending_exams", [])

    structured_answer = {
        "resposta_clinica": state["llm_response"],
        "baseado_em": {
            "sintomas": sintomas,
            "sinais_vitais": sinais_vitais,
            "exames_pendentes": pending_exams,
            "pergunta_medica": state["question"]
        },
        "justificativa_sistematizada": (
            "A resposta foi gerada com base nos dados estruturados do paciente, "
            "na pergunta médica recebida e nas regras de segurança definidas no prompt."
        ),
        "limite_de_atuacao": (
            "O assistente não realiza prescrição medicamentosa e não substitui decisão médica humana."
        )
    }

    return {"structured_answer": structured_answer}


def human_validation(state: MedicalState) -> Dict[str, Any]:
    """Simulação de validação humana.
    Para fins acadêmicos, aprovamos automaticamente casos de risco baixo/moderado
    e sinalizamos revisão reforçada para casos de risco alto.
    """

    risk = state.get("risk_level", "baixo")

    if risk == "alto":
        validation = {
            "status": "revisao_humana_obrigatoria",
            "approved": False,
            "comentario": (
                "Caso classificado como alto risco. Encaminhar imediatamente para avaliação médica presencial."
            )
        }
    else:
        validation = {
            "status": "aprovado_com_supervisao",
            "approved": True,
            "comentario": (
                "Resposta liberada para apoio à decisão clínica, mantendo validação profissional obrigatória."
            )
        }

    return {"human_validation": validation}


def persist_audit_log(state: MedicalState) -> Dict[str, Any]:
    final_output = {
        "patient_id": state["patient_id"],
        "risk_level": state["risk_level"],
        "pending_exams": state.get("pending_exams", []),
        "structured_answer": state["structured_answer"],
        "human_validation": state["human_validation"]
    }

    log_event = {
        "patient_id": state["patient_id"],
        "risk_level": state["risk_level"],
        "patient_data": state["patient_data"],
        "question": state["question"],
        "pending_exams": state.get("pending_exams", []),
        "llm_response": state["llm_response"],
        "structured_answer": state["structured_answer"],
        "human_validation": state["human_validation"]
    }

    log_path = append_log(log_event)

    return {
        "final_output": final_output,
        "log_path": log_path
    }

## Construção do fluxo com LangGraph

Agora conectamos os nós em um fluxo único.

A ordem definida é:

- `triage`
- `check_exams`
- `ask_assistant`
- `build_explainability`
- `human_validation`
- `persist_log`

Esse encadeamento mostra claramente que a recomendação clínica não é entregue de forma imediata.  
Ela passa antes por **explicação** e **validação**, o que fortalece a governança do processo.

In [8]:
builder = StateGraph(MedicalState)

builder.add_node("triage", clinical_triage)
builder.add_node("check_exams", check_pending_exams)
builder.add_node("ask_assistant", ask_medical_assistant)
builder.add_node("build_explainability", build_explainability)
builder.add_node("human_validation", human_validation)
builder.add_node("persist_log", persist_audit_log)

builder.set_entry_point("triage")
builder.add_edge("triage", "check_exams")
builder.add_edge("check_exams", "ask_assistant")
builder.add_edge("ask_assistant", "build_explainability")
builder.add_edge("build_explainability", "human_validation")
builder.add_edge("human_validation", "persist_log")
builder.add_edge("persist_log", END)

graph = builder.compile()

print("✅ Fluxo LangGraph compilado com sucesso.")

✅ Fluxo LangGraph compilado com sucesso.


## Caso de teste

A célula abaixo cria um exemplo de paciente com sinais e sintomas relevantes.  
O objetivo é demonstrar a execução ponta a ponta do fluxo, desde a triagem até o log final.

In [9]:
patient_data = {
    "idade": 52,
    "sexo": "masculino",
    "sintomas": ["febre persistente", "fadiga", "dor no peito"],
    "sinais_vitais": {
        "temperatura": "38.4°C",
        "pressao_arterial": "150/95 mmHg",
        "frequencia_cardiaca": "108 bpm",
        "saturacao": "94%"
    },
    "historico_resumido": [
        "hipertensão arterial sistêmica",
        "episódios prévios de infecção respiratória"
    ],
    "exames_pendentes": ["hemograma completo", "troponina", "raio-x de tórax"]
}

initial_state: MedicalState = {
    "patient_id": "PAC-0001",
    "patient_data": patient_data,
    "question": "Quais próximos passos clínicos devem ser considerados com base no quadro atual?"
}

## Execução do fluxo completo

Nesta etapa o pipeline é executado de ponta a ponta.  
Ao final, exibimos:

- risco clínico
- exames pendentes
- resposta da LLM
- justificativa estruturada
- resultado da validação humana
- caminho do log gerado

In [10]:
result = graph.invoke(initial_state)

print("=== RESULTADO FINAL DO FLUXO ===")
print(f"Paciente: {result['patient_id']}")
print(f"Risco clínico: {result['risk_level']}")
print(f"Exames pendentes: {result.get('pending_exams', [])}")
print()

print("=== RESPOSTA CLÍNICA ===")
print(result["structured_answer"]["resposta_clinica"])
print()

print("=== EXPLAINABILITY ===")
print(json.dumps(result["structured_answer"], ensure_ascii=False, indent=2))
print()

print("=== VALIDAÇÃO HUMANA ===")
print(json.dumps(result["human_validation"], ensure_ascii=False, indent=2))
print()

print("=== LOG GERADO ===")
print(result["log_path"])

=== RESULTADO FINAL DO FLUXO ===
Paciente: PAC-0001
Risco clínico: alto
Exames pendentes: ['hemograma completo', 'troponina', 'raio-x de tórax']

=== RESPOSTA CLÍNICA ===
1. **Análise clínica inicial**  
O paciente apresenta febre persistente, fadiga e dor no peito, com sinais vitais indicando febre (38.4°C), hipertensão (150/95 mmHg), taquicardia (108 bpm) e leve hipoxemia (saturação de 94%). O histórico de hipertensão arterial sistêmica e episódios prévios de infecção respiratória também é relevante. Esses sintomas e sinais podem sugerir uma infecção respiratória, pneumonia ou até mesmo uma condição cardiovascular, como angina ou infarto do miocárdio.

2. **Sugestão clínica**  
Os próximos passos clínicos devem incluir:  
- Realizar os exames pendentes: hemograma completo, troponina e raio-x de tórax.  
- Monitorar os sinais vitais do paciente de forma contínua, especialmente a saturação de oxigênio e a pressão arterial.  
- Considerar a avaliação clínica para possíveis intervenções,

C:\Users\filip\AppData\Local\Temp\ipykernel_21872\510637317.py:7: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  event["timestamp"] = datetime.utcnow().isoformat() + "Z"


## Visualização do último log registrado

Para evidenciar a trilha de auditoria, a célula abaixo lê a última entrada do arquivo de log e exibe seu conteúdo formatado.

In [11]:
with open(result["log_path"], "r", encoding="utf-8") as f:
    linhas = f.readlines()

ultimo_log = json.loads(linhas[-1])

print(json.dumps(ultimo_log, ensure_ascii=False, indent=2))

{
  "patient_id": "PAC-0001",
  "risk_level": "alto",
  "patient_data": {
    "idade": 52,
    "sexo": "masculino",
    "sintomas": [
      "febre persistente",
      "fadiga",
      "dor no peito"
    ],
    "sinais_vitais": {
      "temperatura": "38.4°C",
      "pressao_arterial": "150/95 mmHg",
      "frequencia_cardiaca": "108 bpm",
      "saturacao": "94%"
    },
    "historico_resumido": [
      "hipertensão arterial sistêmica",
      "episódios prévios de infecção respiratória"
    ],
    "exames_pendentes": [
      "hemograma completo",
      "troponina",
      "raio-x de tórax"
    ]
  },
  "question": "Quais próximos passos clínicos devem ser considerados com base no quadro atual?",
  "pending_exams": [
    "hemograma completo",
    "troponina",
    "raio-x de tórax"
  ],
  "llm_response": "1. **Análise clínica inicial**  \nO paciente apresenta febre persistente, fadiga e dor no peito, com sinais vitais indicando febre (38.4°C), hipertensão (150/95 mmHg), taquicardia (108 bp

## Interpretação dos resultados

Com base na execução demonstrada, podemos observar:

- o paciente foi classificado em um **nível de risco**
- os exames pendentes foram considerados no fluxo
- a LLM gerou uma resposta clínica estruturada
- a resposta foi acompanhada de **justificativa**
- houve uma camada explícita de **validação humana**
- toda a execução foi registrada em **log estruturado**

Esses elementos mostram que o sistema foi desenhado para funcionar de forma mais segura e auditável.

## Conclusão

Neste notebook foi implementado um fluxo clínico automatizado com **LangGraph**, integrado a uma **LLM customizada** via **LangChain**.

O principal avanço desta etapa foi adicionar mecanismos de **governança**, tornando o fluxo mais adequado ao contexto hospitalar.  
Em vez de gerar apenas uma resposta textual, o sistema:

- classifica o risco clínico
- consulta o assistente médico
- explicita a base utilizada na resposta
- exige validação humana
- registra toda a execução para auditoria